In [ ]:
# Import the libraries
import pandas as pd  # tables
import numpy as np  # numbers
import textwrap  # wrap long titles
import matplotlib.pyplot as plt  # charts
from google.colab import drive  # Google Drive access

In [ ]:
# Path to the data file (change here if the file moves)
data_path = "/content/drive/My Drive/ghdp_7031_data.csv"  # file on Google Drive
figure_path = "/content/drive/My Drive/fig1_poverty_by_work.png"  # where the chart is saved
# Mount Google Drive
drive.mount("/content/drive")  # asks you to sign in

In [ ]:
# Read the raw file
raw_data = pd.read_csv(data_path)  # load the CSV
# Work on a copy so the raw data stays untouched
data = raw_data.copy()  # copy for all later steps

In [ ]:
# Print the number of rows and columns
print("Shape (rows, columns):", data.shape)  # shape of the table

In [ ]:
# Print the column names
print("Column names:")  # label for the list
print(list(data.columns))  # all column names

In [ ]:
# Build the summary table with 1 decimal
summary_table = data.describe().round(1)  # count, mean, std, min, quartiles, max
# Show the summary table
print(summary_table)  # print the table

In [ ]:
# Fill missing work status with a label
data["fulltime_label"] = data["fulltime_label"].fillna("Did not work last year")  # missing means did not work
# Keep rows that have both poor and asecwt
analysis_data = data.dropna(subset=["poor", "asecwt"]).copy()  # copy so data stays as loaded
# Print how many rows were dropped
print("Rows dropped for missing poor or asecwt:", len(data) - len(analysis_data))  # check the loss
# Print the values poor takes
print("Values of poor:", analysis_data["poor"].drop_duplicates().sort_values().tolist())  # should be 0 and 1
# Stop if poor is not coded 0/1
if not set(analysis_data["poor"].unique()) <= {0, 1}:  # rate math needs 0/1
    raise ValueError("poor is not coded 0/1. Check the codebook before going on.")  # stop with a message

In [ ]:
# Weight each person's poor flag by asecwt
analysis_data["poor_weighted"] = analysis_data["poor"] * analysis_data["asecwt"]  # poor people count by their weight
# Add up poor weight in each year and work status
poor_total = analysis_data.pivot_table(index="year", columns="fulltime_label", values="poor_weighted", aggfunc="sum")  # weighted poor people
# Add up all weight in each year and work status
all_total = analysis_data.pivot_table(index="year", columns="fulltime_label", values="asecwt", aggfunc="sum")  # weighted all people
# Divide to get the poverty rate in percent
poverty_rate = poor_total / all_total * 100  # weighted share who are poor
# Count respondents in each cell, no weights
respondent_counts = analysis_data.pivot_table(index="year", columns="fulltime_label", values="asecwt", aggfunc="count")  # unweighted n
# Print the weighted poverty rate table
print("Poverty rate (% of people, weighted with asecwt)")  # table title
print(poverty_rate.to_string(float_format="{:.1f}".format))  # 1 decimal
# Print the unweighted counts under it
print("\nRespondents in each cell (unweighted)")  # table title
print(respondent_counts.to_string(float_format="{:.0f}".format))  # whole numbers
# Pick the first cell to explain
first_year = poverty_rate.index[0]  # first year row
first_status = poverty_rate.columns[0]  # first work status column
# Explain one cell in plain English
print(f"\nHow to read a cell: in {first_year}, among people whose work status last year was '{first_status}', "
      f"{poverty_rate.loc[first_year, first_status]:.1f}% were poor. "
      "Each person counts by asecwt, the number of people in the US that person stands for. "
      f"This cell rests on {respondent_counts.loc[first_year, first_status]:.0f} respondents.")  # plain sentence

In [ ]:
# Round the table so the chart matches the printed numbers
chart_table = poverty_rate.round(1)  # same 1-decimal values as the printout
# Print the table the chart will show
print("Poverty rate (% of people, weighted with asecwt)")  # table title
print(chart_table.to_string(float_format="{:.1f}".format))  # 1 decimal
# Print the years in the chart so you can check them against the source note
print("Years in chart:", chart_table.index.tolist())  # source note says 2020-2025
# Find the highest and lowest work status in the last year
last_year = chart_table.index[-1]  # latest year in the table
highest_status = chart_table.loc[last_year].idxmax()  # work status with the top rate
lowest_status = chart_table.loc[last_year].idxmin()  # work status with the bottom rate
# Write the title as one sentence that states the finding
chart_title = (f"In {last_year}, the poverty rate was highest for '{highest_status}' "
               f"({chart_table.loc[last_year, highest_status]:.1f}%) and lowest for '{lowest_status}' "
               f"({chart_table.loc[last_year, lowest_status]:.1f}%).")  # finding from the table
# Pick one color per line (blue, orange, aqua, yellow, magenta, green)
line_colors = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]  # fixed order
# Start the figure
fig, ax = plt.subplots(figsize=(9, 6))  # chart size in inches
# Draw one line for each work status
for number, status in enumerate(chart_table.columns):  # loop over columns
    ax.plot(chart_table.index, chart_table[status], color=line_colors[number], linewidth=2, marker="o", markersize=7, label=status)  # the line
    ax.annotate(status, (chart_table.index[-1], chart_table[status].iloc[-1]), xytext=(8, 0), textcoords="offset points", va="center", fontsize=9, color="#0b0b0b")  # label at line end
# Set the y-axis to start at 0 with room at the top
ax.set_ylim(0, chart_table.max().max() * 1.15)  # 0 is always on the axis
# Show every year on the x-axis
ax.set_xticks(chart_table.index)  # one tick per year
# Label the axes with units
ax.set_xlabel("Year")  # x-axis label
ax.set_ylabel("Poverty rate (% of people)")  # y-axis label with units
# Add the title, wrapped to fit
ax.set_title(textwrap.fill(chart_title, 80), loc="left", fontsize=12)  # finding in one sentence
# Add a light grid and remove the top and right lines
ax.grid(axis="y", color="#dddddd", linewidth=0.8)  # soft horizontal grid
ax.spines[["top", "right"]].set_visible(False)  # cleaner frame
# Add the legend
ax.legend(title="Work status last year", frameon=False)  # legend for the lines
# Add the source note under the chart
fig.text(0.01, -0.02, "Source: CPS ASEC 2020-2025 via IPUMS, weighted with asecwt", fontsize=9, color="#52514e")  # source note
# Save the chart to Google Drive
plt.savefig(figure_path, dpi=200, bbox_inches="tight")  # PNG file
# Show the chart
plt.show()  # display in Colab